# Clase 103 — Keras Functional API y Subclassing

Cuando el modelo no es una pila lineal —skip connections, multi-input, multi-output,
capas compartidas— usamos la **Functional API** (estilo grafo de capas). Y cuando
necesitamos flujo de control **dinámico**, el **Subclassing** (`class MyModel(Model)`).

Requiere: `tensorflow` / `keras`. Se ejecuta en Colab con GPU.

## 1. Functional API: cada capa se llama como función

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

# output = Capa(...)(input)  -> permite cualquier DAG
inputs = keras.Input(shape=(8,))
x = layers.Dense(32, activation="relu")(inputs)
x = layers.Dense(16, activation="relu")(x)
outputs = layers.Dense(1)(x)
modelo = keras.Model(inputs=inputs, outputs=outputs, name="mlp_funcional")
print("capas:", [l.name for l in modelo.layers])
print("params:", modelo.count_params())

## 2. Wide & Deep (Cheng et al. 2016)

Dos rutas: una **wide** (input directo a la salida) y una **deep** (Dense apiladas)
que se combinan con `Concatenate`.

In [ ]:
input_ = keras.Input(shape=(8,), name="entrada")
# ruta deep
hidden1 = layers.Dense(30, activation="relu")(input_)
hidden2 = layers.Dense(30, activation="relu")(hidden1)
# ruta wide: la entrada salta directo a la fusión
concat = layers.Concatenate()([input_, hidden2])
output = layers.Dense(1)(concat)
wide_deep = keras.Model(inputs=input_, outputs=output, name="wide_and_deep")
wide_deep.compile(optimizer="adam", loss="mse")
wide_deep.summary()

## 3. Multi-output: una representación compartida, dos cabezas

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

X, y = fetch_california_housing(return_X_y=True)
mediana = np.median(y)
y_caro = (y > mediana).astype("float32")     # etiqueta binaria auxiliar
X_tr, X_te, y_tr, y_te, yc_tr, yc_te = train_test_split(
    X, y, y_caro, test_size=0.2, random_state=42)

ent = keras.Input(shape=(X.shape[1],))
h = layers.Dense(32, activation="relu")(ent)
h = layers.Dense(32, activation="relu")(h)               # tronco compartido
salida_reg = layers.Dense(1, name="precio")(h)           # regresión
salida_cls = layers.Dense(1, activation="sigmoid", name="caro")(h)  # binaria

multi = keras.Model(inputs=ent, outputs=[salida_reg, salida_cls])
multi.compile(optimizer="adam",
              loss={"precio": "mse", "caro": "binary_crossentropy"},
              loss_weights={"precio": 0.7, "caro": 0.3},
              metrics={"caro": "accuracy"})
multi.fit(X_tr, {"precio": y_tr, "caro": yc_tr}, epochs=5, verbose=0)
print("modelo multi-output entrenado; salidas:", [o.name for o in multi.outputs])

## 4. Capa compartida (siamesa): la misma instancia en dos entradas

In [ ]:
encoder = layers.Dense(16, activation="relu")   # UNA sola instancia
in_a = keras.Input(shape=(8,)); in_b = keras.Input(shape=(8,))
emb_a = encoder(in_a)      # mismos pesos
emb_b = encoder(in_b)      # mismos pesos
distancia = layers.Concatenate()([emb_a, emb_b])
salida = layers.Dense(1, activation="sigmoid")(distancia)   # "same / different"
siamesa = keras.Model(inputs=[in_a, in_b], outputs=salida)
print("el encoder comparte pesos entre ambas ramas:",
      siamesa.get_layer(encoder.name).count_params(), "parámetros")

## 5. Subclassing: `ResBlock` con skip connection

Heredamos de `keras.layers.Layer`, creamos las capas en `__init__` y definimos el
forward en `call()`. La skip connection es `Add()([x, F(x)])`.

In [ ]:
class ResBlock(keras.layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.dense1 = layers.Dense(units, activation="relu")
        self.dense2 = layers.Dense(units)
        self.add = layers.Add()
        self.act = layers.Activation("relu")

    def call(self, inputs, training=False):
        z = self.dense1(inputs)
        z = self.dense2(z)
        z = self.add([inputs, z])      # conexión residual (preserva dimensión)
        return self.act(z)

modelo_res = keras.Sequential([
    keras.Input(shape=(16,)),
    layers.Dense(32, activation="relu"),
    ResBlock(32),
    ResBlock(32),
    layers.Dense(1),
])
modelo_res.compile(optimizer="adam", loss="mse")
print("modelo con ResBlocks:", modelo_res.count_params(), "parámetros")

## 6. Subclassing de `Model`: forward dinámico con `training`

In [ ]:
class MLPDinamico(keras.Model):
    def __init__(self, n_clases=3, **kwargs):
        super().__init__(**kwargs)
        self.h1 = layers.Dense(32, activation="relu")
        self.drop = layers.Dropout(0.3)
        self.out = layers.Dense(n_clases, activation="softmax")

    def call(self, inputs, training=False):
        x = self.h1(inputs)
        x = self.drop(x, training=training)   # dropout solo en entrenamiento
        return self.out(x)

modelo_sub = MLPDinamico(n_clases=3)
modelo_sub.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
                   metrics=["accuracy"])
# subclassing: guardar solo pesos y reconstruir la arquitectura en código
print("modelo subclass listo; el flag training controla el Dropout.")

## Ejercicios

1. **Wide & Deep**: entrená el modelo de la sección 2 sobre California Housing y
   compará su MAE con un MLP secuencial simple.
2. **Multi-output**: reportá simultáneamente el MAE de `precio` y la accuracy de
   `caro`. Experimentá con distintos `loss_weights`.
3. **Siamesa**: generá pares (misma/distinta clase) y entrená la red a decidir
   "same/different" con el encoder compartido.
4. **ResBlock vs sin skip**: compará la loss de un modelo con `ResBlock` contra el
   mismo sin la conexión residual.

## Conclusiones

- La **Functional API** modela cualquier DAG: `out = Capa()(in)`, luego `Model(inputs, outputs)`.
- Permite **multi-input**, **multi-output** y **capas compartidas** (misma instancia = mismos pesos).
- Las **skip connections** (`Add()`) preservan dimensión; `Concatenate()` fusiona modalidades.
- El **Subclassing** da control dinámico en `call(inputs, training)`, a costa de serialización más difícil.
- Regla práctica: **Sequential -> Functional (por defecto) -> Subclassing** solo si hace falta flujo dinámico.